# Phase 1 — IEEE 802.1CB FRER over Wi-Fi: Latency Analysis

This notebook analyzes one-way UDP latency measurements collected across five
network conditions, each repeated over five runs of 10 000 packets at 1 ms
intervals.

| Label | Path(s) | FRER |
|-------|---------|------|
| **RAW-A** | wlan0 only (VNI 101) | No |
| **RAW-B** | wlx only (VNI 102) | No |
| **FRER-A** | wlan0 only (VNI 101) | Yes (replicate + eliminate) |
| **FRER-B** | wlx only (VNI 102) | Yes (replicate + eliminate) |
| **FRER-AB** | Both wlan0 + wlx | Yes (replicate + eliminate) |

In [ ]:
import pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

## 1. Load data

Set `BASE` to the directory that contains the five condition folders
(`rawA`, `rawB`, `frerA`, `frerB`, `frerAB`).  Each folder holds
`run*_recv.csv` files with columns `seq,send_epoch_ns,recv_epoch_ns,latency_ns`.

In [ ]:
BASE = pathlib.Path(".")  # adjust if data is elsewhere

CONDITIONS = ["rawA", "rawB", "frerA", "frerB", "frerAB"]
LABELS = {
    "rawA": "RAW-A",
    "rawB": "RAW-B",
    "frerA": "FRER-A",
    "frerB": "FRER-B",
    "frerAB": "FRER-AB",
}
COLORS = {
    "rawA": "#1f77b4",
    "rawB": "#ff7f0e",
    "frerA": "#2ca02c",
    "frerB": "#d62728",
    "frerAB": "#9467bd",
}
EXPECTED_PACKETS = 10_000


def load_condition(name: str) -> pd.DataFrame:
    """Load all receiver CSVs for a condition and return a combined DataFrame."""
    folder = BASE / name
    files = sorted(folder.glob("run*_recv.csv")) or sorted(folder.glob("run*.csv"))
    if not files:
        print(f"  WARNING: no CSV files found in {folder}")
        return pd.DataFrame()
    frames = []
    for i, f in enumerate(files, 1):
        df = pd.read_csv(f)
        df["run"] = i
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


data = {}
for cond in CONDITIONS:
    df = load_condition(cond)
    if not df.empty:
        df["latency_ms"] = df["latency_ns"] / 1e6
        data[cond] = df
        print(f"{LABELS[cond]:10s}  {len(df):>6d} packets across {df['run'].nunique()} runs")

print(f"\nLoaded {len(data)}/{len(CONDITIONS)} conditions.")

## 2. Summary statistics

In [ ]:
rows = []
for cond in CONDITIONS:
    if cond not in data:
        continue
    df = data[cond]
    lat = df["latency_ms"]
    total_sent = df["run"].nunique() * EXPECTED_PACKETS
    loss = total_sent - len(df)
    rows.append({
        "Condition": LABELS[cond],
        "Packets": len(df),
        "Loss": loss,
        "Loss %": f"{loss / total_sent * 100:.3f}",
        "Min (ms)": f"{lat.min():.3f}",
        "Mean (ms)": f"{lat.mean():.3f}",
        "Median (ms)": f"{lat.median():.3f}",
        "P95 (ms)": f"{lat.quantile(0.95):.3f}",
        "P99 (ms)": f"{lat.quantile(0.99):.3f}",
        "P99.9 (ms)": f"{lat.quantile(0.999):.3f}",
        "Max (ms)": f"{lat.max():.3f}",
        "Std (ms)": f"{lat.std():.3f}",
    })

summary = pd.DataFrame(rows)
summary

## 3. Per-run summary

In [ ]:
per_run_rows = []
for cond in CONDITIONS:
    if cond not in data:
        continue
    for run, g in data[cond].groupby("run"):
        lat = g["latency_ms"]
        per_run_rows.append({
            "Condition": LABELS[cond],
            "Run": run,
            "Packets": len(g),
            "Loss": EXPECTED_PACKETS - len(g),
            "Mean (ms)": round(lat.mean(), 3),
            "P50 (ms)": round(lat.median(), 3),
            "P99 (ms)": round(lat.quantile(0.99), 3),
            "Max (ms)": round(lat.max(), 3),
        })

per_run = pd.DataFrame(per_run_rows)
per_run

## 4. CDF — All conditions

In [ ]:
fig, ax = plt.subplots()
for cond in CONDITIONS:
    if cond not in data:
        continue
    lat = np.sort(data[cond]["latency_ms"].values)
    cdf = np.arange(1, len(lat) + 1) / len(lat)
    ax.plot(lat, cdf, label=LABELS[cond], color=COLORS[cond], linewidth=1.2)

ax.set_xlabel("One-way latency (ms)")
ax.set_ylabel("CDF")
ax.set_title("Cumulative Distribution — All Conditions")
ax.legend()
ax.set_xlim(left=0)
ax.yaxis.set_major_formatter(ticker.PercentFormatter(1.0))
fig.tight_layout()
plt.show()

### 4a. CDF — zoomed to 99th percentile

In [ ]:
fig, ax = plt.subplots()
p99_max = 0
for cond in CONDITIONS:
    if cond not in data:
        continue
    lat = np.sort(data[cond]["latency_ms"].values)
    cdf = np.arange(1, len(lat) + 1) / len(lat)
    ax.plot(lat, cdf, label=LABELS[cond], color=COLORS[cond], linewidth=1.2)
    p99_max = max(p99_max, np.percentile(lat, 99))

ax.set_xlabel("One-way latency (ms)")
ax.set_ylabel("CDF")
ax.set_title("CDF — Zoomed to P99")
ax.legend()
ax.set_xlim(left=0, right=p99_max * 1.3)
ax.yaxis.set_major_formatter(ticker.PercentFormatter(1.0))
fig.tight_layout()
plt.show()

## 5. Box plots — latency distribution

In [ ]:
plot_data = []
plot_labels = []
plot_colors = []
for cond in CONDITIONS:
    if cond not in data:
        continue
    plot_data.append(data[cond]["latency_ms"].values)
    plot_labels.append(LABELS[cond])
    plot_colors.append(COLORS[cond])

fig, ax = plt.subplots()
bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,
                showfliers=False, whis=[5, 95])
for patch, color in zip(bp["boxes"], plot_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.set_ylabel("One-way latency (ms)")
ax.set_title("Latency Distribution (whiskers at P5–P95, no outliers)")
fig.tight_layout()
plt.show()

## 6. Violin plots — full distribution shape

In [ ]:
fig, ax = plt.subplots()
parts = ax.violinplot(plot_data, showmedians=True, showextrema=False)
for i, pc in enumerate(parts["bodies"]):
    pc.set_facecolor(plot_colors[i])
    pc.set_alpha(0.6)

ax.set_xticks(range(1, len(plot_labels) + 1))
ax.set_xticklabels(plot_labels)
ax.set_ylabel("One-way latency (ms)")
ax.set_title("Latency Distribution — Violin Plot")
fig.tight_layout()
plt.show()

## 7. Latency time series — per run

In [ ]:
n_conds = len(data)
fig, axes = plt.subplots(n_conds, 1, figsize=(12, 3.2 * n_conds), sharex=True)
if n_conds == 1:
    axes = [axes]

for ax, cond in zip(axes, [c for c in CONDITIONS if c in data]):
    for run, g in data[cond].groupby("run"):
        ax.plot(g["seq"].values, g["latency_ms"].values,
                linewidth=0.4, alpha=0.7, label=f"Run {run}")
    ax.set_ylabel("Latency (ms)")
    ax.set_title(LABELS[cond])
    ax.legend(fontsize=8, ncol=5, loc="upper right")

axes[-1].set_xlabel("Packet sequence number")
fig.suptitle("Latency Time Series", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## 8. Comparisons

### 8a. RAW vs FRER (single path)

In [ ]:
pairs = [("rawA", "frerA"), ("rawB", "frerB")]

for raw, frer in pairs:
    if raw not in data or frer not in data:
        continue
    raw_lat = data[raw]["latency_ms"]
    frer_lat = data[frer]["latency_ms"]
    print(f"=== {LABELS[raw]} vs {LABELS[frer]} ===")
    print(f"  Mean   : {raw_lat.mean():.3f} ms → {frer_lat.mean():.3f} ms  "
          f"(Δ {frer_lat.mean() - raw_lat.mean():+.3f} ms)")
    print(f"  Median : {raw_lat.median():.3f} ms → {frer_lat.median():.3f} ms  "
          f"(Δ {frer_lat.median() - raw_lat.median():+.3f} ms)")
    print(f"  P99    : {raw_lat.quantile(0.99):.3f} ms → {frer_lat.quantile(0.99):.3f} ms  "
          f"(Δ {frer_lat.quantile(0.99) - raw_lat.quantile(0.99):+.3f} ms)")
    raw_loss = data[raw]["run"].nunique() * EXPECTED_PACKETS - len(data[raw])
    frer_loss = data[frer]["run"].nunique() * EXPECTED_PACKETS - len(data[frer])
    print(f"  Loss   : {raw_loss} → {frer_loss}")
    print()

### 8b. Single-path FRER vs Dual-path FRER

In [ ]:
for single in ["frerA", "frerB"]:
    dual = "frerAB"
    if single not in data or dual not in data:
        continue
    s = data[single]["latency_ms"]
    d = data[dual]["latency_ms"]
    print(f"=== {LABELS[single]} vs {LABELS[dual]} ===")
    print(f"  Mean   : {s.mean():.3f} ms → {d.mean():.3f} ms  "
          f"(Δ {d.mean() - s.mean():+.3f} ms)")
    print(f"  Median : {s.median():.3f} ms → {d.median():.3f} ms  "
          f"(Δ {d.median() - s.median():+.3f} ms)")
    print(f"  P99    : {s.quantile(0.99):.3f} ms → {d.quantile(0.99):.3f} ms  "
          f"(Δ {d.quantile(0.99) - s.quantile(0.99):+.3f} ms)")
    s_loss = data[single]["run"].nunique() * EXPECTED_PACKETS - len(data[single])
    d_loss = data[dual]["run"].nunique() * EXPECTED_PACKETS - len(data[dual])
    print(f"  Loss   : {s_loss} → {d_loss}")
    print()

### 8c. CDF comparison — RAW-A vs FRER-A vs FRER-AB

In [ ]:
compare = ["rawA", "frerA", "frerAB"]

fig, ax = plt.subplots()
for cond in compare:
    if cond not in data:
        continue
    lat = np.sort(data[cond]["latency_ms"].values)
    cdf = np.arange(1, len(lat) + 1) / len(lat)
    ax.plot(lat, cdf, label=LABELS[cond], color=COLORS[cond], linewidth=1.5)

ax.set_xlabel("One-way latency (ms)")
ax.set_ylabel("CDF")
ax.set_title("RAW-A vs FRER-A vs FRER-AB")
ax.legend()
ax.set_xlim(left=0)
ax.yaxis.set_major_formatter(ticker.PercentFormatter(1.0))
fig.tight_layout()
plt.show()

## 9. Packet loss per run

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(5)  # 5 runs
width = 0.15
offsets = np.linspace(-2 * width, 2 * width, len(data))

for i, cond in enumerate([c for c in CONDITIONS if c in data]):
    losses = []
    for run in range(1, 6):
        g = data[cond][data[cond]["run"] == run]
        losses.append(EXPECTED_PACKETS - len(g))
    ax.bar(x + offsets[i], losses, width, label=LABELS[cond], color=COLORS[cond], alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels([f"Run {i}" for i in range(1, 6)])
ax.set_ylabel("Packets lost")
ax.set_title("Packet Loss per Run")
ax.legend()
fig.tight_layout()
plt.show()